## 1. Setup

In [1]:
from transformers import AutoTokenizer
import pandas as pd
from typing import List, Dict
import numpy as np

/home/cong/miniconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load tokenizer
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

## 2. Subword Tokenization - Ví dụ minh họa

### Vấn đề cốt lõi:
- Một số từ → 1 token
- Một số từ → nhiều subword tokens

Điều này tạo ra **label alignment problem**

In [3]:
# Ví dụ 1: Từ đơn giản
word1 = "Washington"
tokens1 = tokenizer.tokenize(word1)
print(f"Word: {word1}")
print(f"Tokens: {tokens1}")
print(f"Number of tokens: {len(tokens1)}")
print()

Word: Washington
Tokens: ['▁Washington']
Number of tokens: 1



In [4]:
# Ví dụ 2: Từ phức tạp hơn
word2 = "playing"
tokens2 = tokenizer.tokenize(word2)
print(f"Word: {word2}")
print(f"Tokens: {tokens2}")
print(f"Number of tokens: {len(tokens2)}")
print()

# VẤN ĐỀ: Nếu "playing" có label B-VERB,
# thì "play" và "##ing" được gán label gì?

Word: playing
Tokens: ['▁playing']
Number of tokens: 1



In [5]:
# Ví dụ 3: Nhiều từ khác nhau
test_words = [
    "Obama",
    "Washington",
    "playing",
    "organization",
    "beautifully",
    "COVID-19",
    "New York"
]

print("=" * 80)
print("SUBWORD TOKENIZATION EXAMPLES")
print("=" * 80)

for word in test_words:
    tokens = tokenizer.tokenize(word)
    print(f"\n{word:20s} → {tokens}")
    print(f"{'':20s}   ({len(tokens)} token(s))")

SUBWORD TOKENIZATION EXAMPLES

Obama                → ['▁Obama']
                       (1 token(s))

Washington           → ['▁Washington']
                       (1 token(s))

playing              → ['▁playing']
                       (1 token(s))

organization         → ['▁organization']
                       (1 token(s))

beautifully          → ['▁beautiful', 'ly']
                       (2 token(s))

COVID-19             → ['▁CO', 'VID', '-19']
                       (3 token(s))

New York             → ['▁New', '▁York']
                       (2 token(s))


## 3. Label Alignment Problem

### Scenario:
```
Sentence: "Barack Obama visited Washington"
Word-level labels: [B-PER, I-PER, O, B-LOC]
```

### Sau tokenization:
```
Tokens: ["Bar", "##ack", "Obama", "visited", "Washington"]
Token-level labels: [?, ?, ?, ?, ?]
```

**Câu hỏi:** Label nào cho "##ack"?

In [6]:
# Demo label alignment problem
sentence = "Barack Obama visited Washington"
words = sentence.split()
word_labels = ["B-PER", "I-PER", "O", "B-LOC"]

print("=" * 80)
print("LABEL ALIGNMENT PROBLEM")
print("=" * 80)
print("\nOriginal (Word-level):")
for word, label in zip(words, word_labels):
    print(f"  {word:15s} → {label}")

print("\nAfter Tokenization:")
tokenized = tokenizer(sentence, add_special_tokens=False)
tokens = tokenizer.convert_ids_to_tokens(tokenized['input_ids'])
for i, token in enumerate(tokens):
    print(f"  Token {i}: {token:15s} → ???")

LABEL ALIGNMENT PROBLEM

Original (Word-level):
  Barack          → B-PER
  Obama           → I-PER
  visited         → O
  Washington      → B-LOC

After Tokenization:
  Token 0: ▁Barack         → ???
  Token 1: ▁Obama          → ???
  Token 2: ▁visite         → ???
  Token 3: d               → ???
  Token 4: ▁Washington     → ???


## 4. Alignment Strategies

### Strategy A: First Token Only
- Gán label cho token **đầu tiên** của mỗi word
- Các token còn lại → **IGNORE** (không tính loss)

```
playing (B-VERB)
  ↓
play    → B-VERB
##ing   → IGNORE (-100)
```

### Strategy B: Label Propagation
- Token đầu tiên: giữ nguyên label
- Các token tiếp theo: chuyển B-X → I-X

```
Barack (B-PER)
  ↓
Bar     → B-PER
##ack   → I-PER
```

### Strategy C: Custom Alignment
- Tùy theo tokenizer/model cụ thể
- Sử dụng `word_ids()` từ tokenizer

## 5. Implementation - Strategy A (First Token Only)

In [7]:
def align_labels_strategy_a(
    tokenized_inputs,
    word_labels: List[str],
    label_to_id: Dict[str, int],
    ignore_index: int = -100
) -> List[int]:
    """
    Strategy A: First Token Only
    
    - Token đầu tiên của word → label gốc
    - Token tiếp theo của cùng word → IGNORE
    - Special tokens ([CLS], [SEP]) → IGNORE
    
    Args:
        tokenized_inputs: Output từ tokenizer
        word_labels: List các label theo word
        label_to_id: Dictionary mapping label → id
        ignore_index: Giá trị cho IGNORE (mặc định -100)
        
    Returns:
        List các label ids theo token
    """
    word_ids = tokenized_inputs.word_ids()
    aligned_labels = []
    previous_word_id = None
    
    for word_id in word_ids:
        # Special tokens → IGNORE
        if word_id is None:
            aligned_labels.append(ignore_index)
        
        # Token đầu tiên của word → label gốc
        elif word_id != previous_word_id:
            label = word_labels[word_id]
            aligned_labels.append(label_to_id[label])
        
        # Token tiếp theo của cùng word → IGNORE
        else:
            aligned_labels.append(ignore_index)
        
        previous_word_id = word_id
    
    return aligned_labels

In [8]:
# Demo Strategy A
sentence = "Barack Obama visited Washington"
words = sentence.split()
word_labels = ["B-PER", "I-PER", "O", "B-LOC"]

# Create label mapping
unique_labels = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

print("Label mapping:")
print(label_to_id)
print()

# Tokenize
tokenized = tokenizer(
    sentence,
    is_split_into_words=False,
    return_offsets_mapping=True
)

# Align labels
aligned_label_ids = align_labels_strategy_a(
    tokenized,
    word_labels,
    label_to_id,
    ignore_index=-100
)

# Display results
tokens = tokenizer.convert_ids_to_tokens(tokenized['input_ids'])

print("=" * 80)
print("STRATEGY A: FIRST TOKEN ONLY")
print("=" * 80)
print(f"\n{'Token':<20} {'Label ID':<15} {'Label'}")
print("-" * 80)
for token, label_id in zip(tokens, aligned_label_ids):
    if label_id == -100:
        label_str = "IGNORE"
    else:
        label_str = id_to_label[label_id]
    print(f"{token:<20} {str(label_id):<15} {label_str}")

Label mapping:
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6}

STRATEGY A: FIRST TOKEN ONLY

Token                Label ID        Label
--------------------------------------------------------------------------------
<s>                  -100            IGNORE
▁Barack              1               B-PER
▁Obama               2               I-PER
▁visite              0               O
d                    -100            IGNORE
▁Washington          5               B-LOC
</s>                 -100            IGNORE


## 6. Implementation - Strategy B (Label Propagation)

In [9]:
def align_labels_strategy_b(
    tokenized_inputs,
    word_labels: List[str],
    label_to_id: Dict[str, int],
    ignore_index: int = -100
) -> List[int]:
    """
    Strategy B: Label Propagation
    
    - Token đầu tiên của word → label gốc
    - Token tiếp theo: B-X → I-X (propagate)
    - Special tokens → IGNORE
    
    Args:
        tokenized_inputs: Output từ tokenizer
        word_labels: List các label theo word
        label_to_id: Dictionary mapping label → id
        ignore_index: Giá trị cho IGNORE
        
    Returns:
        List các label ids theo token
    """
    word_ids = tokenized_inputs.word_ids()
    aligned_labels = []
    previous_word_id = None
    
    for word_id in word_ids:
        # Special tokens → IGNORE
        if word_id is None:
            aligned_labels.append(ignore_index)
        
        # Token đầu tiên của word → label gốc
        elif word_id != previous_word_id:
            label = word_labels[word_id]
            aligned_labels.append(label_to_id[label])
        
        # Token tiếp theo → propagate (B-X → I-X)
        else:
            label = word_labels[word_id]
            
            # Nếu label bắt đầu bằng B-, chuyển thành I-
            if label.startswith("B-"):
                label = "I-" + label[2:]
            
            aligned_labels.append(label_to_id[label])
        
        previous_word_id = word_id
    
    return aligned_labels

In [10]:
# Demo Strategy B
aligned_label_ids_b = align_labels_strategy_b(
    tokenized,
    word_labels,
    label_to_id,
    ignore_index=-100
)

print("=" * 80)
print("STRATEGY B: LABEL PROPAGATION")
print("=" * 80)
print(f"\n{'Token':<20} {'Label ID':<15} {'Label'}")
print("-" * 80)
for token, label_id in zip(tokens, aligned_label_ids_b):
    if label_id == -100:
        label_str = "IGNORE"
    else:
        label_str = id_to_label[label_id]
    print(f"{token:<20} {str(label_id):<15} {label_str}")

STRATEGY B: LABEL PROPAGATION

Token                Label ID        Label
--------------------------------------------------------------------------------
<s>                  -100            IGNORE
▁Barack              1               B-PER
▁Obama               2               I-PER
▁visite              0               O
d                    0               O
▁Washington          5               B-LOC
</s>                 -100            IGNORE


## 7. So sánh hai strategies

In [11]:
# So sánh trực quan
comparison_data = []

for token, label_a, label_b in zip(tokens, aligned_label_ids, aligned_label_ids_b):
    label_a_str = "IGNORE" if label_a == -100 else id_to_label[label_a]
    label_b_str = "IGNORE" if label_b == -100 else id_to_label[label_b]
    
    comparison_data.append({
        "Token": token,
        "Strategy A": label_a_str,
        "Strategy B": label_b_str,
        "Difference": "✓" if label_a_str == label_b_str else "✗"
    })

comparison_df = pd.DataFrame(comparison_data)

print("=" * 80)
print("COMPARISON: STRATEGY A vs STRATEGY B")
print("=" * 80)
print(comparison_df.to_string(index=False))

COMPARISON: STRATEGY A vs STRATEGY B
      Token Strategy A Strategy B Difference
        <s>     IGNORE     IGNORE          ✓
    ▁Barack      B-PER      B-PER          ✓
     ▁Obama      I-PER      I-PER          ✓
    ▁visite          O          O          ✓
          d     IGNORE          O          ✗
▁Washington      B-LOC      B-LOC          ✓
       </s>     IGNORE     IGNORE          ✓


## 8. Ignore Index và Loss Calculation

### Tại sao cần IGNORE?

Khi training, loss function chỉ tính trên các token có label thật:

$$
L = -\sum_{i: y_i \neq \text{IGNORE}} \log P(y_i | x)
$$

### IGNORE không có nghĩa:
- Token đó không tồn tại ❌
- Token đó không được dùng trong model ❌

### IGNORE có nghĩa:
- Label của token đó không được dùng để tính loss ✓
- Tránh penalize model trên các token không có ground-truth label rõ ràng ✓

In [12]:
# Demo: Ignore Index in Loss
import torch
import torch.nn.functional as F

# Giả sử predictions và labels
num_labels = len(unique_labels)
predictions = torch.randn(len(tokens), num_labels)  # Random predictions
labels_a = torch.tensor(aligned_label_ids)  # Strategy A labels

print("=" * 80)
print("LOSS CALCULATION WITH IGNORE INDEX")
print("=" * 80)

# Loss WITHOUT ignore
# (This would be wrong - counting special tokens and subwords)
loss_without_ignore = F.cross_entropy(
    predictions,
    labels_a.clamp(min=0),  # Clamp to avoid -100
    reduction='mean'
)

# Loss WITH ignore (correct way)
loss_with_ignore = F.cross_entropy(
    predictions,
    labels_a,
    ignore_index=-100,  # Ignore special tokens and continuation subwords
    reduction='mean'
)

print(f"\nTokens total: {len(tokens)}")
print(f"Tokens with IGNORE: {(labels_a == -100).sum().item()}")
print(f"Tokens contributing to loss: {(labels_a != -100).sum().item()}")
print(f"\nLoss (without ignore): {loss_without_ignore.item():.4f}")
print(f"Loss (with ignore): {loss_with_ignore.item():.4f}")
print("\n⚠️  Always use ignore_index=-100 in loss calculation!")

LOSS CALCULATION WITH IGNORE INDEX

Tokens total: 7
Tokens with IGNORE: 3
Tokens contributing to loss: 4

Loss (without ignore): 1.9449
Loss (with ignore): 2.0153

⚠️  Always use ignore_index=-100 in loss calculation!


## 9. Practical Example: Full Pipeline

In [13]:
def tokenize_and_align_labels(
    examples,
    tokenizer,
    label_to_id,
    strategy="first_token",
    max_length=128
):
    """
    Complete function để tokenize và align labels.
    Dùng trong training pipeline.
    
    Args:
        examples: Dictionary với keys 'tokens' và 'ner_tags'
        tokenizer: HuggingFace tokenizer
        label_to_id: Label mapping
        strategy: 'first_token' hoặc 'propagate'
        max_length: Max sequence length
        
    Returns:
        Tokenized inputs với aligned labels
    """
    # Tokenize
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=max_length,
        padding="max_length"
    )
    
    # Align labels
    labels = []
    for i, label_seq in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_label_seq = []
        previous_word_id = None
        
        for word_id in word_ids:
            # Special tokens
            if word_id is None:
                aligned_label_seq.append(-100)
            
            # First token of word
            elif word_id != previous_word_id:
                aligned_label_seq.append(label_seq[word_id])
            
            # Continuation tokens
            else:
                if strategy == "first_token":
                    aligned_label_seq.append(-100)
                elif strategy == "propagate":
                    # B-X → I-X
                    label_id = label_seq[word_id]
                    # (Simplified - need proper B-/I- handling)
                    aligned_label_seq.append(label_id)
            
            previous_word_id = word_id
        
        labels.append(aligned_label_seq)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [14]:
# Test với example data
example_data = {
    "tokens": [
        ["Barack", "Obama", "visited", "Washington"],
        ["Apple", "CEO", "Tim", "Cook"]
    ],
    "ner_tags": [
        [1, 2, 0, 5],  # B-PER, I-PER, O, B-LOC
        [3, 0, 1, 2]   # B-ORG, O, B-PER, I-PER
    ]
}

# Apply tokenization and alignment
result = tokenize_and_align_labels(
    example_data,
    tokenizer,
    label_to_id,
    strategy="first_token"
)

print("=" * 80)
print("TOKENIZATION & ALIGNMENT PIPELINE OUTPUT")
print("=" * 80)
print(f"\nInput IDs shape: {len(result['input_ids'])} x {len(result['input_ids'][0])}")
print(f"Labels shape: {len(result['labels'])} x {len(result['labels'][0])}")
print(f"\nFirst example tokens:")
print(tokenizer.convert_ids_to_tokens(result['input_ids'][0][:15]))
print(f"\nFirst example labels:")
print(result['labels'][0][:15])

TOKENIZATION & ALIGNMENT PIPELINE OUTPUT

Input IDs shape: 2 x 128
Labels shape: 2 x 128

First example tokens:
['<s>', '▁Barack', '▁Obama', '▁visite', 'd', '▁Washington', '</s>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']

First example labels:
[-100, 1, 2, 0, -100, 5, -100, -100, -100, -100, -100, -100, -100, -100, -100]


## 10. Analysis: Impact của Strategy Choice

In [15]:
def analyze_strategy_impact(sentence, word_labels, tokenizer, label_to_id):
    """
    Phân tích impact của strategy choice.
    """
    tokenized = tokenizer(
        sentence,
        is_split_into_words=False,
        return_offsets_mapping=True
    )
    
    # Apply both strategies
    labels_a = align_labels_strategy_a(tokenized, word_labels, label_to_id)
    labels_b = align_labels_strategy_b(tokenized, word_labels, label_to_id)
    
    # Count differences
    total_tokens = len(labels_a)
    ignored_a = sum(1 for l in labels_a if l == -100)
    ignored_b = sum(1 for l in labels_b if l == -100)
    different = sum(1 for a, b in zip(labels_a, labels_b) if a != b)
    
    print("=" * 80)
    print("STRATEGY IMPACT ANALYSIS")
    print("=" * 80)
    print(f"\nTotal tokens: {total_tokens}")
    print(f"\nStrategy A (First Token Only):")
    print(f"  - Tokens with IGNORE: {ignored_a} ({ignored_a/total_tokens*100:.1f}%)")
    print(f"  - Tokens with label: {total_tokens - ignored_a}")
    print(f"\nStrategy B (Propagation):")
    print(f"  - Tokens with IGNORE: {ignored_b} ({ignored_b/total_tokens*100:.1f}%)")
    print(f"  - Tokens with label: {total_tokens - ignored_b}")
    print(f"\nDifferences: {different} tokens ({different/total_tokens*100:.1f}%)")
    print(f"\n💡 Insight:")
    print(f"   Strategy B sử dụng nhiều token hơn cho training")
    print(f"   → Có thể học tốt hơn, nhưng cũng có thể overfit")

# Run analysis
analyze_strategy_impact(
    "Barack Obama visited Washington",
    ["B-PER", "I-PER", "O", "B-LOC"],
    tokenizer,
    label_to_id
)

STRATEGY IMPACT ANALYSIS

Total tokens: 7

Strategy A (First Token Only):
  - Tokens with IGNORE: 3 (42.9%)
  - Tokens with label: 4

Strategy B (Propagation):
  - Tokens with IGNORE: 2 (28.6%)
  - Tokens with label: 5

Differences: 1 tokens (14.3%)

💡 Insight:
   Strategy B sử dụng nhiều token hơn cho training
   → Có thể học tốt hơn, nhưng cũng có thể overfit


## 11. Recommendations

### Strategy A (First Token Only)
**Ưu điểm:**
- Đơn giản, dễ implement
- Tránh noise từ subword boundaries
- Phổ biến trong literature

**Nhược điểm:**
- Bỏ đi thông tin từ continuation tokens
- Có thể kém hơn với từ dài

### Strategy B (Propagation)
**Ưu điểm:**
- Sử dụng nhiều token hơn cho training
- Có thể học tốt hơn entity boundaries

**Nhược điểm:**
- Phức tạp hơn
- Có thể overfit trên subword pattern